# Predicting Stellar Class - Playground Series S6E6

This notebook provides a baseline solution for the Stellar Classification competition using XGBoost.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

## Load Data

In [ ]:
# Load the datasets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain columns: {train.columns.tolist()}")
print(f"\nClass distribution:")
print(train['class'].value_counts())

## Data Exploration

In [ ]:
# Display basic statistics
train.describe()

In [ ]:
# Check for missing values
print("Missing values in train:")
print(train.isnull().sum())
print("\nMissing values in test:")
print(test.isnull().sum())

In [ ]:
# Visualize class distribution
plt.figure(figsize=(10, 6))
train['class'].value_counts().plot(kind='bar')
plt.title('Distribution of Stellar Classes')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## Prepare Features

In [ ]:
# Define features and target
target_col = 'class'
id_col = 'id'

# Get feature columns (all except id and target)
feature_cols = [col for col in train.columns if col not in [id_col, target_col]]

print(f"Number of features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

# Encode target labels
target_map = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
inv_target_map = {v: k for k, v in target_map.items()}

# Prepare data
X_train = train[feature_cols]
y_train = train[target_col].map(target_map)
X_test = test[feature_cols]

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")

## Train XGBoost Model with Cross-Validation

In [ ]:
# XGBoost parameters
xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'max_depth': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'seed': SEED,
    'n_jobs': -1,
    'tree_method': 'hist'
}

# Cross-validation setup
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Store out-of-fold predictions and test predictions
oof_preds = np.zeros((len(X_train), 3))
test_preds = np.zeros((len(X_test), 3))
cv_scores = []

print("Starting Cross-Validation...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f"Fold {fold}/{N_FOLDS}")
    
    # Split data
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Train model
    model = xgb.XGBClassifier(**xgb_params, n_estimators=500)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    # Predictions
    val_preds = model.predict_proba(X_val)
    oof_preds[val_idx] = val_preds
    test_preds += model.predict_proba(X_test) / N_FOLDS
    
    # Calculate balanced accuracy
    val_pred_labels = np.argmax(val_preds, axis=1)
    fold_score = balanced_accuracy_score(y_val, val_pred_labels)
    cv_scores.append(fold_score)
    
    print(f"Fold {fold} Balanced Accuracy: {fold_score:.6f}\n")

# Overall CV score
oof_pred_labels = np.argmax(oof_preds, axis=1)
overall_score = balanced_accuracy_score(y_train, oof_pred_labels)

print(f"\n{'='*50}")
print(f"Mean CV Balanced Accuracy: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")
print(f"Overall OOF Balanced Accuracy: {overall_score:.6f}")
print(f"{'='*50}")

## Evaluate Results

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_train, oof_pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['GALAXY', 'QSO', 'STAR'],
            yticklabels=['GALAXY', 'QSO', 'STAR'])
plt.title(f'Confusion Matrix (OOF)\nBalanced Accuracy: {overall_score:.6f}')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-class Accuracy:")
for i, class_name in enumerate(['GALAXY', 'QSO', 'STAR']):
    mask = (y_train == i)
    class_acc = (oof_pred_labels[mask] == i).mean()
    print(f"{class_name:>6s}: {class_acc:.6f}")

## Create Submission

In [ ]:
# Create submission
test_pred_labels = np.argmax(test_preds, axis=1)
submission = sample_submission.copy()
submission['class'] = test_pred_labels
submission['class'] = submission['class'].map(inv_target_map)

# Save submission
submission.to_csv('submission.csv', index=False)

print("Submission file created: submission.csv")
print(f"\nSubmission preview:")
print(submission.head(10))
print(f"\nSubmission class distribution:")
print(submission['class'].value_counts())

## Submit to Kaggle

To submit your results to the competition, run the following command in your terminal:

```bash
kaggle competitions submit -c playground-series-s6e6 -f submission.csv -m "XGBoost baseline submission"
```